# InspectAI — PatchCore Feature-Layer Ablation

Compare `layer2`, `layer3`, and `layer2 + layer3` with **800 coreset patches** and **256 × 256** input images. Uses the original 176 normal training / 44 normal validation split and *synthetic* anomalies for validation. **No official test images are used.**

## 1. Setup

All configurations use the same image paths, preprocessing, pretrained ResNet18, synthetic validation examples, and coreset sampling settings. Each layer configuration gets its **own memory bank**.

In [ ]:
from pathlib import Path
import json
import time

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights
from torchvision.models.feature_extraction import create_feature_extractor
from sklearn.metrics import roc_auc_score, average_precision_score

DATA_DIR = Path('../data/raw/metal_nut').resolve()
SPLIT_FILE = Path('../results/normal_split.json').resolve()
RESULTS_DIR = Path('../results').resolve()
FIGURES_DIR = RESULTS_DIR / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
SEED = 42
IMAGE_SIZE = 256
BATCH_SIZE = 8
CORESET_SIZE = 800
NUM_CANDIDATES = 4000
PROJECTION_DIM = 64
DISTANCE_CHUNK_SIZE = 1024
PIXEL_EVAL_SIZE = 128
CONFIGURATIONS = ('layer2', 'layer3', 'layer2+layer3')

np.random.seed(SEED)
torch.manual_seed(SEED)
assert DATA_DIR.is_dir(), DATA_DIR
assert SPLIT_FILE.is_file(), SPLIT_FILE
with SPLIT_FILE.open() as f:
    split = json.load(f)
normal_dir = DATA_DIR / 'train' / 'good'
train_paths = [normal_dir / name for name in split['train']]
val_paths = [normal_dir / name for name in split['validation']]
assert len(train_paths) == 176 and len(val_paths) == 44
assert set(train_paths).isdisjoint(val_paths)
assert all(path.is_file() for path in train_paths + val_paths)
print('Device:', DEVICE, 'Train:', len(train_paths), 'Validation:', len(val_paths))

## 2. Preprocessing and CNN feature extraction

A frozen ImageNet-pretrained ResNet18 produces `layer2` (32 × 32) and `layer3` (16 × 16) maps. For the combined model, `layer3` is upsampled to `layer2`'s grid. We apply the same 3 × 3 local average pooling as in our earlier PatchCore implementation.

In [ ]:
preprocess = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class PathDataset(Dataset):
    def __init__(self, paths):
        self.paths = list(paths)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        with Image.open(self.paths[idx]) as im:
            return preprocess(im.convert('RGB'))

train_loader = DataLoader(PathDataset(train_paths), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

backbone = resnet18(weights=ResNet18_Weights.DEFAULT)
feature_extractor = create_feature_extractor(
    backbone, return_nodes={'layer2': 'layer2', 'layer3': 'layer3'}
).to(DEVICE).eval()
for param in feature_extractor.parameters():
    param.requires_grad_(False)

@torch.inference_mode()
def extract_embedding(images, configuration):
    features = feature_extractor(images.to(DEVICE))
    layer2 = F.avg_pool2d(features['layer2'], 3, stride=1, padding=1)
    layer3 = F.avg_pool2d(features['layer3'], 3, stride=1, padding=1)
    if configuration == 'layer2':
        return layer2
    if configuration == 'layer3':
        return layer3
    if configuration == 'layer2+layer3':
        aligned = F.interpolate(layer3, size=layer2.shape[-2:], mode='bilinear', align_corners=False)
        return torch.cat([layer2, aligned], dim=1)
    raise ValueError(configuration)

batch = next(iter(train_loader))
for config in CONFIGURATIONS:
    print(config, tuple(extract_embedding(batch, config).shape))

## 3. Build separate normal patch memory banks

Only the **176 training images** contribute features. Select 4,000 random candidate patches then use projected greedy farthest-point sampling to retain 800 patches. The same random seed is used across configurations; feature dimensions and grid sizes differ.

In [ ]:
@torch.inference_mode()
def build_patch_bank(loader, configuration):
    chunks = []
    for images in loader:
        embeddings = extract_embedding(images, configuration)
        chunks.append(embeddings.permute(0, 2, 3, 1).reshape(-1, embeddings.shape[1]).cpu())
    return torch.cat(chunks, dim=0)

def select_coreset(bank):
    generator = torch.Generator().manual_seed(SEED)
    candidate_idx = torch.randperm(len(bank), generator=generator)[:NUM_CANDIDATES]
    candidates = bank[candidate_idx]
    projection = torch.randn(candidates.shape[1], PROJECTION_DIM, generator=generator) / np.sqrt(PROJECTION_DIM)
    projected = candidates @ projection
    min_sq = torch.full((len(candidates),), float('inf'))
    chosen = []
    current = 0
    for _ in range(CORESET_SIZE):
        chosen.append(current)
        distances = (projected - projected[current]).square().sum(dim=1)
        min_sq = torch.minimum(min_sq, distances)
        min_sq[chosen] = -1
        current = int(torch.argmax(min_sq))
    return candidates[chosen].contiguous()

banks = {}
selection_times = {}
for config in CONFIGURATIONS:
    start = time.perf_counter()
    full_bank = build_patch_bank(train_loader, config)
    banks[config] = select_coreset(full_bank)
    selection_times[config] = time.perf_counter() - start
    print(config, 'full:', tuple(full_bank.shape), 'coreset:', tuple(banks[config].shape),
          'seconds:', round(selection_times[config], 2))
    del full_bank

## 4. Generate synthetic validation anomalies

From every held-out normal image, create one synthetic scratch and one color defect, while retaining an unchanged normal copy. We reuse the exact generator design from `06_patchcore_hyperparameter_tuning.ipynb` for consistency. Synthetic defects are only a proxy for real anomalies.

In [ ]:
def make_synthetic(im, kind, seed):
    rng = np.random.default_rng(seed)
    image = im.convert('RGB').resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.BILINEAR)
    modified = image.copy()
    mask = Image.new('L', image.size, 0)
    draw = ImageDraw.Draw(modified)
    mask_draw = ImageDraw.Draw(mask)
    if kind == 'scratch':
        cx, cy = int(rng.integers(65, 190)), int(rng.integers(65, 190))
        dx, dy = int(rng.integers(22, 57)), int(rng.integers(-48, 49))
        width = int(rng.integers(3, 8))
        line = [(cx-dx//2, cy-dy//2), (cx+dx//2, cy+dy//2)]
        color = (int(rng.integers(12, 45)),)*3
        draw.line(line, fill=color, width=width)
        mask_draw.line(line, fill=255, width=width)
    elif kind == 'color':
        cx, cy = int(rng.integers(70, 187)), int(rng.integers(70, 187))
        radius = int(rng.integers(11, 25))
        box = (cx-radius, cy-radius, cx+radius, cy+radius)
        draw.ellipse(box, fill=(145, 42, 58))
        mask_draw.ellipse(box, fill=255)
    else:
        raise ValueError(kind)
    return modified, np.asarray(mask, dtype=np.uint8)

val_images, val_masks, val_labels, val_kinds = [], [], [], []
for i, path in enumerate(val_paths):
    with Image.open(path) as raw:
        original = raw.convert('RGB').resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.BILINEAR)
        val_images.append(original)
        val_masks.append(np.zeros((IMAGE_SIZE, IMAGE_SIZE), dtype=np.uint8))
        val_labels.append(0)
        val_kinds.append('good')
        for j, kind in enumerate(('scratch', 'color')):
            modified, mask = make_synthetic(original, kind, SEED + 100*i + j)
            val_images.append(modified)
            val_masks.append(mask)
            val_labels.append(1)
            val_kinds.append(kind)
val_labels = np.asarray(val_labels)
val_masks = np.stack(val_masks)
assert len(val_images) == 132 and val_labels.sum() == 88
print('Synthetic validation:', len(val_images), 'images; anomalous:', val_labels.sum())

fig, axes = plt.subplots(1, 3, figsize=(9, 3))
for ax, i in zip(axes, [0, 1, 2]):
    ax.imshow(val_images[i]); ax.set_title(val_kinds[i]); ax.axis('off')
plt.tight_layout(); plt.show()

## 5. Cache validation features for all three configurations

Each configuration gets spatial embeddings extracted from the **same validation images**. We never fit the memory bank using these images.

In [ ]:
class PILDataset(Dataset):
    def __init__(self, images):
        self.images = images

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return preprocess(self.images[idx])

val_loader = DataLoader(PILDataset(val_images), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
val_embeddings = {}
for config in CONFIGURATIONS:
    chunks = []
    with torch.inference_mode():
        for batch in val_loader:
            chunks.append(extract_embedding(batch, config).cpu())
    val_embeddings[config] = torch.cat(chunks, dim=0)
    print(config, tuple(val_embeddings[config].shape))

## 6. Evaluate synthetic-validation anomaly detection

For each patch, use Euclidean distance to its nearest normal memory patch. The maximum patch distance gives the image score. Spatial maps are bilinearly resized to 128 × 128 for pixel evaluation; ground-truth masks use nearest-neighbor resize.

**Caution:** `layer3` uses a coarser 16 × 16 grid, while `layer2` and the combination use 32 × 32; localization resolution is therefore part of this ablation.

In [ ]:
eval_masks = np.stack([
    np.asarray(Image.fromarray(mask).resize((PIXEL_EVAL_SIZE, PIXEL_EVAL_SIZE), Image.Resampling.NEAREST)) > 0
    for mask in val_masks
])

@torch.inference_mode()
def evaluate_configuration(config, bank):
    on_device = bank.to(DEVICE).float()
    all_scores, all_maps = [], []
    start = time.perf_counter()
    for embedding in val_embeddings[config].split(BATCH_SIZE):
        embedding = embedding.to(DEVICE)
        batch, channels, height, width = embedding.shape
        patches = embedding.permute(0, 2, 3, 1).reshape(-1, channels)
        min_distances = []
        for piece in patches.split(DISTANCE_CHUNK_SIZE):
            min_distances.append(torch.cdist(piece.float(), on_device).min(dim=1).values)
        maps = torch.cat(min_distances).reshape(batch, height, width)
        all_scores.append(maps.flatten(1).max(1).values.cpu().numpy())
        all_maps.append(maps.cpu())
    if DEVICE.type == 'mps':
        torch.mps.synchronize()
    seconds = time.perf_counter() - start
    scores = np.concatenate(all_scores)
    maps = torch.cat(all_maps)
    upsampled = F.interpolate(
        maps.unsqueeze(1), size=(PIXEL_EVAL_SIZE, PIXEL_EVAL_SIZE),
        mode='bilinear', align_corners=False
    ).squeeze(1).numpy()
    normal_scores = scores[val_labels == 0]
    return {
        'synthetic_image_auroc': float(roc_auc_score(val_labels, scores)),
        'synthetic_image_ap': float(average_precision_score(val_labels, scores)),
        'synthetic_pixel_auroc': float(roc_auc_score(eval_masks.ravel(), upsampled.ravel())),
        'synthetic_pixel_ap': float(average_precision_score(eval_masks.ravel(), upsampled.ravel())),
        'normal_p95_score': float(np.percentile(normal_scores, 95)),
        'scoring_seconds_per_image': float(seconds / len(scores)),
    }

results = {}
for config in CONFIGURATIONS:
    results[config] = evaluate_configuration(config, banks[config])
    print('\n' + config)
    for metric, value in results[config].items():
        print(f'  {metric}: {value:.5f}')

results_file = RESULTS_DIR / 'patchcore_layer_ablation.json'
with results_file.open('w') as f:
    json.dump({'protocol': 'synthetic-validation-only', 'coreset_size': CORESET_SIZE,
               'input_size': IMAGE_SIZE, 'results': results}, f, indent=2)
print('Saved:', results_file)

## 7. Visual comparison and interpretation

Plot key metrics. The three variants must be compared using the same synthetic validation examples, but higher synthetic scores do not guarantee better performance on real MVTec defects.

In [ ]:
labels = list(CONFIGURATIONS)
x = np.arange(len(labels))
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(x, [results[k]['synthetic_image_auroc'] for k in labels])
axes[0].set_title('Synthetic image AUROC')
axes[0].set_ylim(0, 1.05)
axes[1].bar(x, [results[k]['synthetic_pixel_ap'] for k in labels])
axes[1].set_title('Synthetic pixel AP')
axes[1].set_ylim(0, 1.05)
for ax in axes:
    ax.set_xticks(x, labels)
    ax.set_ylabel('Score')
plt.tight_layout()
figure_file = FIGURES_DIR / 'patchcore_layer_ablation.png'
fig.savefig(figure_file, dpi=140, bbox_inches='tight')
plt.show()

best_synthetic = max(labels, key=lambda k: (
    results[k]['synthetic_image_auroc'], results[k]['synthetic_pixel_ap']
))
print('Highest synthetic-validation image AUROC:', best_synthetic)
print('Existing demo model remains unchanged: models/patchcore_final_bank.pt')

## 8. Track runs in DagsHub / MLflow

Results and figure are saved **locally first**. This cell optionally creates three new MLflow runs. If a network timeout occurs, local files are preserved and the evaluation need not be repeated.

In [ ]:
LOG_TO_DAGSHUB = True
if LOG_TO_DAGSHUB:
    try:
        import dagshub
        import mlflow

        dagshub.init(repo_owner='lchkh23', repo_name='inspectAI', mlflow=True)
        mlflow.set_experiment('metal-nut-anomaly-detection')

        for config in CONFIGURATIONS:
            with mlflow.start_run(run_name=f'patchcore-synthetic-layer-ablation-{config.replace("+", "-")}'):
                mlflow.log_params({
                    'model': 'simplified_patchcore',
                    'backbone': 'resnet18',
                    'feature_layers': config,
                    'input_size': IMAGE_SIZE,
                    'coreset_size': CORESET_SIZE,
                    'candidate_count': NUM_CANDIDATES,
                    'projection_dim': PROJECTION_DIM,
                    'seed': SEED,
                    'train_images': len(train_paths),
                    'validation_normal_images': len(val_paths),
                    'synthetic_anomalies_per_normal': 2,
                    'validation_type': 'synthetic',
                    'official_test_used': False,
                    'embedding_dim': banks[config].shape[1],
                    'spatial_grid': int(val_embeddings[config].shape[-1]),
                })
                mlflow.log_metrics(results[config])
                mlflow.log_metric('bank_construction_seconds', float(selection_times[config]))
                mlflow.log_artifact(str(results_file), artifact_path='results')
                mlflow.log_artifact(str(figure_file), artifact_path='figures')
        print('Logged three feature-layer ablation runs.')
    except Exception as exc:
        print('DagsHub logging failed, but local results are safe:', type(exc).__name__, str(exc)[:300])
        try:
            if mlflow.active_run() is not None:
                mlflow.end_run(status='FAILED')
        except Exception:
            pass

## 9. Conclusions (fill after running)

- Compare the three configurations on **synthetic** validation image AUROC and pixel AP.
- Note the cost of memory-bank construction and different embedding dimensions.
- Single-layer `layer3` has a coarser spatial grid, which may limit localization.
- This is **not** a new independent evaluation on the official MVTec test set. The model used by Streamlit remains the originally saved 800-patch `layer2 + layer3` model until an independently justified update.